# 1 таск

In [ ]:
# !pip install -q --upgrade transformers==4.57.1 trl==0.24.0 peft==0.17.1 accelerate==1.10.1 bitsandbytes==0.47.0 datasets==3.6.0

In [8]:
import random
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

from transformers import set_seed

import json

import numpy as np
import torch

SEED = 42
DATA_DIR = Path("/kaggle/input/datasets/lambdaderta/red-cu-task/ml-olympiad-red-task")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [9]:
def load_jsonl(name):
    path = DATA_DIR / "data" / name
    with open(path, encoding="utf-8") as file:
        return [json.loads(line) for line in file]

kid_adult = load_jsonl("kid_adult.jsonl")
good_bad = load_jsonl("good_bad.jsonl")
test_style = load_jsonl("public_test_style.jsonl")
test_quality = load_jsonl("public_test_quality.jsonl")

In [ ]:
kid_adult[1]

In [ ]:
good_bad[0]

In [ ]:
test_style[0]

In [ ]:
test_quality[0]

In [ ]:
# ! pip install -U bitsandbytes

In [10]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map="auto",
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [ ]:
model

In [ ]:
examples = []

for row in kid_adult:
    example = {
        "prompt": [
            {"role": "user", "content": row["prompt"]}
        ],
        "completion": [
            {"role": "assistant", "content": row["kid"]}
        ],
    }

    examples.append(example)

sft_data = Dataset.from_list(examples)

In [ ]:
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

In [ ]:
model

In [ ]:
# ! pip install trl

In [11]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="sft_style",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_length=512,
    fp16=True,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    loss_type="nll",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=sft_data,
    processing_class=tokenizer,
)

NameError: name 'sft_data' is not defined

In [ ]:
trainer.train()

In [ ]:
adapter_path = "/kaggle/working/sft_style_adapter"

trainer.save_model(adapter_path)
tokenizer.save_pretrained(adapter_path)

In [12]:
import pickle

model.config.use_cache = True
model.eval()

with open(DATA_DIR / "metrics" / "style_clf.pkl", "rb") as file:
    style_clf = pickle.load(file)

answers = []

for row in test_style:
    messages = [
        {"role": "user", "content": row["prompt"]}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    start = inputs["input_ids"].shape[1]
    new_tokens = output[0][start:]

    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
    answers.append(answer)

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking c

KeyboardInterrupt: 

In [ ]:
style_clf.keys()

## P_simple подсчет

In [13]:
from scipy.sparse import hstack

vectors = []

for vec in style_clf["vecs"]:
    vector = vec.transform(answers)
    vectors.append(vector)

features = hstack(vectors)

probabilities = style_clf["clf"].predict_proba(features)[:, 1]
p_simple = float(np.mean(probabilities))

print("P_simple:", p_simple)

P_simple: 0.1998601396868253


# 2 таск

In [14]:
model.load_adapter(
    adapter_path,
    adapter_name="reference",
    is_trainable=False,
)

model.set_adapter("default")
model.config.use_cache = False

NameError: name 'adapter_path' is not defined

In [15]:
dpo_examples = []

for row in kid_adult:
    example = {
        "prompt": [
            {"role": "user", "content": row["prompt"]}
        ],
        "chosen": [
            {"role": "assistant", "content": row["kid"]}
        ],
        "rejected": [
            {"role": "assistant", "content": row["adult"]}
        ],
    }

    dpo_examples.append(example)

dpo_data = Dataset.from_list(dpo_examples)

In [16]:
from trl import DPOConfig, DPOTrainer

dpo_args = DPOConfig(
    output_dir="dpo_style",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
    fp16=True,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    model_adapter_name="default",
    ref_adapter_name="reference",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_data,
    processing_class=tokenizer,
)

Extracting prompt in train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1489 [00:00<?, ? examples/s]

ValueError: You cannot perform fine-tuning on purely quantized models. Please attach trainable adapters on top of the quantized model to correctly perform fine-tuning. Please see: https://huggingface.co/docs/transformers/peft for more details

In [ ]:
dpo_trainer.train()

In [ ]:
dpo_path = "/kaggle/working/dpo_style_adapter"

trainer.save_model(dpo_path)
tokenizer.save_pretrained(dpo_path)

In [17]:
import json

public_test_style = []

with open(f"{DATA_DIR}/data/public_test_style.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        row = json.loads(line)
        public_test_style.append(row)

print(len(public_test_style))
print(public_test_style[0])

50
{'prompt': 'Почему продавцы в магазинах делают скидки и распродажи? Ответ: Чтобы быстрее распродать старые товары, освободить место для новых коллекций или привлечь больше покупателей.', 'kid': 'Магазину нужно освободить полки, чтобы поставить новые игрушки и вещи. Поэтому старые товары продают дешевле, как будто кричат: «Забирайте скорее!». Это здорово помогает быстро всё разобрать и порадовать покупателей.', 'adult': 'Продавцы делают распродажи для ускорения оборачиваемости товаров и освобождения складских запасов перед закупкой новых коллекций. Маркетинговое снижение цены стимулирует покупательский спрос и позволяет конвертировать неликвидные активы в живые деньги, избегая замораживания капитала.', 'fact': 'Скидки и распродажи используются для ускорения сбыта остатков, расчистки складов под новые поступления и стимуляции спроса.'}


In [18]:
import torch
from transformers import set_seed
from tqdm.auto import tqdm

set_seed(42)

dpo_model = trainer.model
dpo_model.set_adapter("default")
dpo_model.eval()

answers = []

for row in tqdm(public_test_style):
    messages = [
        {"role": "user", "content": row["prompt"]}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(dpo_model.device)

    with torch.no_grad():
        output = dpo_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = output[0, inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    answers.append(answer)

AttributeError: 'NoneType' object has no attribute 'model'

## Метрика после дпо

In [19]:
import joblib
from scipy.sparse import hstack

style_data = joblib.load("/kaggle/input/datasets/lambdaderta/red-cu-task/ml-olympiad-red-task/metrics/style_clf.pkl")

clf = style_data["clf"]
vecs = style_data["vecs"]

vectors = []

for vec in vecs:
    transformed_answers = vec.transform(answers)
    vectors.append(transformed_answers)

answer_vectors = hstack(vectors)

probabilities = clf.predict_proba(answer_vectors)[:, 1]
p_simple_dpo = float(np.mean(probabilities))

print("DPO style P_simple:", p_simple_dpo)
print(answers[0])

DPO style P_simple: 0.1998601396868253
Правильный ответ: **Чтобы быстрее распродать старые товары, освободить место для новых коллекций или привлечь больше покупателей.**

Это объяснение корректно и полно. Продавцы используют скидки и распродажи для нескольких ключевых целей:

1. **Ускорения продаж старых товаров** — товары, которые уже не востребованы, могут быть проданы дешевле, чтобы освободить место и бюджет.
2. **Освобождение места** — магазины постоянно обновляют ассортимент, поэтому скидки помогают убрать устаревшие позиции, чтобы внести новые, актуальные продукты.
3. **Привлечение покупателей** — скидки создают предложение, которое стимулирует покупателей прийти в магазин, особенно если они ищут выгодные цены или сэкономить.

Таким образом, ответ — **правильный и полный**. ✅


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking c

# 3 таск

In [20]:
import gc
import torch

trainer = None
model = None
dpo_model = None
answers = None

gc.collect()
torch.cuda.empty_cache()

In [ ]:
good_bad = []

with open(f"{DATA_DIR}/data/good_bad.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        row = json.loads(line)
        good_bad.append(row)

len(good_bad)

In [ ]:
from datasets import Dataset

rm_rows = []

for row in good_bad:
    chosen_messages = [
        {
            "role": "user",
            "content": row["instruction"],
        },
        {
            "role": "assistant",
            "content": row["chosen"],
        },
    ]

    rejected_messages = [
        {
            "role": "user",
            "content": row["instruction"],
        },
        {
            "role": "assistant",
            "content": row["rejected"],
        },
    ]

    new_row = {
        "chosen": chosen_messages,
        "rejected": rejected_messages,
    }

    rm_rows.append(new_row)

rm_data = Dataset.from_list(rm_rows)

In [ ]:
rm_data = rm_data.train_test_split(
    test_size=0.1,
    seed=42,
)

rm_train = rm_data["train"]
rm_val = rm_data["test"]

rm_train[0]

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

rm_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

rm_model.config.pad_token_id = tokenizer.pad_token_id
rm_model.config.problem_type = "regression"
rm_model.config.use_cache = False

In [ ]:
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

rm_model = prepare_model_for_kbit_training(rm_model)

rm_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    modules_to_save=["score"],
)

rm_model = get_peft_model(
    rm_model,
    rm_lora_config,
)

rm_model.print_trainable_parameters()

In [ ]:
from trl import RewardConfig, RewardTrainer

rm_args = RewardConfig(
    output_dir="/kaggle/working/rm_checkpoints",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    max_length=512,

    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    optim="paged_adamw_8bit",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,

    logging_steps=25,
    save_total_limit=2,

    seed=42,
    data_seed=42,
    report_to="none",
)

In [ ]:
rm_trainer = RewardTrainer(
    model=rm_model,
    args=rm_args,
    train_dataset=rm_train,
    eval_dataset=rm_val,
    processing_class=tokenizer,
)

In [ ]:
rm_trainer.train()

In [ ]:
rm_path = "/kaggle/working/reward_model_adapter"

rm_trainer.save_model(rm_path)
tokenizer.save_pretrained(rm_path)

In [ ]:
import json

public_test_quality = []

with open(
    f"{DATA_DIR}/data/public_test_quality.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        row = json.loads(line)
        public_test_quality.append(row)

len(public_test_quality)

In [ ]:
import torch

rm_model = rm_trainer.model
rm_model.eval()

def get_reward(instruction, answer):
    messages = [
        {
            "role": "user",
            "content": instruction,
        },
        {
            "role": "assistant",
            "content": answer,
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=False,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        truncation=True,
        max_length=512,
    ).to(rm_model.device)

    with torch.no_grad():
        output = rm_model(**inputs)

    score = output.logits[0, 0].float().item()

    return score

## метрика 3 таск

In [ ]:
correct = 0
margins = []

for row in tqdm(public_test_quality):
    chosen_score = get_reward(
        row["prompt"],
        row["chosen"],
    )

    rejected_score = get_reward(
        row["prompt"],
        row["rejected"],
    )

    if chosen_score > rejected_score:
        correct += 1

    margins.append(chosen_score - rejected_score)

rm_accuracy = correct / len(public_test_quality)
mean_margin = float(np.mean(margins))

print("Public RM accuracy:", rm_accuracy)
print("Public mean margin:", mean_margin)

# 4 таска да что так много черный лвл быстрее решается

In [ ]:
rm_trainer = None
rm_model = None

gc.collect()
torch.cuda.empty_cache()

In [21]:
quality_rows = []

for row in good_bad:
    new_row = {
        "prompt": [
            {
                "role": "user",
                "content": row["instruction"],
            }
        ],
        "chosen": [
            {
                "role": "assistant",
                "content": row["chosen"],
            }
        ],
        "rejected": [
            {
                "role": "assistant",
                "content": row["rejected"],
            }
        ],
    }

    quality_rows.append(new_row)

quality_data = Dataset.from_list(quality_rows)

In [22]:
quality_data = quality_data.train_test_split(
    test_size=0.1,
    seed=42,
)

quality_train = quality_data["train"]
quality_val = quality_data["test"]

quality_train[0]

{'prompt': [{'role': 'user',
   'content': 'Почему одеяло сохраняет нас в тепле? Ткань не дает телу нагреться, она просто ловит тепло, которое вырабатывает наше тело, и не пускает холодный воздух.'}],
 'chosen': [{'role': 'assistant',
   'content': 'Ты абсолютно прав, одеяло работает как тёплый домик, который сам не греет, а бережёт твоё тепло! Между ниточками ткани прячется воздух, который не пускает тепло нашего тела наружу.'}],
 'rejected': [{'role': 'assistant',
   'content': 'Ой, ты такой умный, что сам всё понял, но давай я расскажу подробнее! Знаешь, одеяло вообще-то не живое и в нём нет батарейки, поэтому оно совсем не умеет греть, как солнышко. Просто представь, что одеяло   это такая большая и пушистая стена, которая накрыла тебя сверху. И вот ты лежишь под ней и вырабатываешь тепло, а оно берёт и не пускает его на волю гулять.'}]}

In [23]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
DPO_STYLE_PATH = "/kaggle/working/dpo_style_adapter"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

base_model = prepare_model_for_kbit_training(
    base_model,
)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [24]:
from peft import (
    PeftModel,
    prepare_model_for_kbit_training,
)

quality_model = PeftModel.from_pretrained(
    base_model,
    DPO_STYLE_PATH,
    adapter_name="default",
    is_trainable=True,
)

ValueError: Can't find 'adapter_config.json' at '/kaggle/working/dpo_style_adapter'

In [ ]:
quality_model.load_adapter(
    DPO_STYLE_PATH,
    adapter_name="reference",
    is_trainable=False,
)

quality_model.set_adapter("default")
quality_model.config.use_cache = False

quality_model.print_trainable_parameters()

In [ ]:
tokenizer.padding_side = "left"

In [25]:
from trl import DPOConfig, DPOTrainer

quality_args = DPOConfig(
    output_dir="/kaggle/working/dpo_quality_checkpoints",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=5e-6,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    beta=0.1,

    max_length=512,
    max_prompt_length=256,

    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    optim="paged_adamw_8bit",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    logging_steps=25,

    model_adapter_name="default",
    ref_adapter_name="reference",

    seed=42,
    data_seed=42,
    report_to="none",
)

In [ ]:
quality_trainer = DPOTrainer(
    model=quality_model,
    ref_model=None,

    args=quality_args,

    train_dataset=quality_train,
    eval_dataset=quality_val,

    processing_class=tokenizer,
)

In [ ]:
quality_trainer.train()

In [ ]:
quality_path = "/kaggle/working/dpo_quality_adapter"

quality_trainer.save_model(quality_path)
tokenizer.save_pretrained(quality_path)

In [ ]:
import torch.nn.functional as F

quality_model = quality_trainer.model
quality_model.set_adapter("default")
quality_model.eval()

def mean_answer_logprob(prompt, answer):
    prompt_messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        add_generation_prompt=True,
        tokenize=True,
    )

    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
    )["input_ids"]

    all_ids = prompt_ids + answer_ids

    input_ids = torch.tensor(
        [all_ids],
        device=quality_model.device,
    )

    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        output = quality_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

    logits = output.logits[0]

    answer_logits = logits[
        len(prompt_ids) - 1 : len(all_ids) - 1
    ]

    answer_targets = input_ids[
        0,
        len(prompt_ids) :
    ]

    log_probs = F.log_softmax(
        answer_logits.float(),
        dim=-1,
    )

    answer_log_probs = log_probs.gather(
        dim=1,
        index=answer_targets.unsqueeze(1),
    ).squeeze(1)

    return answer_log_probs.mean().item()

# метрика 4 таск

In [ ]:
correct = 0
margins = []

for row in tqdm(public_test_quality):
    chosen_logprob = mean_answer_logprob(
        row["prompt"],
        row["chosen"],
    )

    rejected_logprob = mean_answer_logprob(
        row["prompt"],
        row["rejected"],
    )

    if chosen_logprob > rejected_logprob:
        correct += 1

    margins.append(
        chosen_logprob - rejected_logprob
    )

quality_accuracy = correct / len(public_test_quality)
quality_mean_margin = float(np.mean(margins))

print("DPO quality accuracy:", quality_accuracy)
print("DPO quality mean margin:", quality_mean_margin)

# 5 последняя уаррааааа наконец то я решаю это всю ночь ну зачем так долго то

In [26]:
quality_trainer = None
quality_model = None
base_model = None

gc.collect()
torch.cuda.empty_cache()

In [28]:
from transformers import AutoModelForCausalLM
from peft import PeftModel, prepare_model_for_kbit_training

DPO_STYLE_PATH = "/kaggle/working/dpo_style_adapter"
# DPO_STYLE_PATH = "/kaggle/input/datasets/lambdaderta/backup-kaggle-padla/dpo_style_adapter" # Это у меня кагл умер вот но как бы оно костыль да

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

base_model = prepare_model_for_kbit_training(base_model)

simpo_model = PeftModel.from_pretrained(
    base_model,
    DPO_STYLE_PATH,
    is_trainable=True,
)

simpo_model.config.use_cache = False
tokenizer.padding_side = "left"

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [29]:
from trl import CPOConfig, CPOTrainer

simpo_args = CPOConfig(
    output_dir="/kaggle/working/simpo_checkpoints",

    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-6,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    loss_type="simpo",
    cpo_alpha=0.0,
    beta=2.0,
    simpo_gamma=1.0,

    max_length=512,
    max_prompt_length=256,

    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=25,

    seed=42,
    data_seed=42,
    report_to="none",
)

In [30]:
simpo_trainer = CPOTrainer(
    model=simpo_model,
    args=simpo_args,
    train_dataset=quality_train,
    eval_dataset=quality_val,
    processing_class=tokenizer,
)

/usr/local/lib/python3.12/dist-packages/trl/trainer/cpo_trainer.py:148: UserWarning: This trainer will soon be moved to trl.experimental and is a candidate for removal. If you rely on it and want it to remain, please share your comments here: https://github.com/huggingface/trl/issues/4223. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  warnings.warn(


Map:   0%|          | 0/2003 [00:00<?, ? examples/s]

Map:   0%|          | 0/2003 [00:00<?, ? examples/s]

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

Map:   0%|          | 0/2003 [00:00<?, ? examples/s]

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

In [31]:
simpo_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Runtime,Samples Per Second,Steps Per Second,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/rejected,Logps/chosen,Logits/rejected,Logits/chosen,Nll Loss
1,1.698200,nan,101.458100,2.198000,1.104000,-3.308630,nan,0.000000,nan,nan,-1.654315,nan,-2.286404,0.000000


TrainOutput(global_step=126, training_loss=0.3369376470181301, metrics={'train_runtime': 2506.7052, 'train_samples_per_second': 0.799, 'train_steps_per_second': 0.05, 'total_flos': 0.0, 'train_loss': 0.3369376470181301, 'epoch': 1.0})

In [34]:
simpo_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.L

In [39]:
import json

DATA_PATH = "/kaggle/input/datasets/lambdaderta/red-cu-task/ml-olympiad-red-task"

public_test_quality = []

with open(DATA_PATH + "/data/public_test_quality.jsonl", encoding="utf-8") as file:
    for line in file:
        public_test_quality.append(json.loads(line))

In [40]:
import torch

def mean_answer_logprob(model, prompt, answer):
    messages = [
        {"role": "user", "content": prompt}
    ]

    prompt_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
    )

    answer_ids = tokenizer(
        answer,
        add_special_tokens=False,
    )["input_ids"]

    all_ids = prompt_ids + answer_ids

    input_ids = torch.tensor(
        [all_ids],
        device=model.device,
    )

    with torch.inference_mode():
        logits = model(input_ids=input_ids).logits

    logits = logits[:, :-1].float()
    labels = input_ids[:, 1:]

    log_probs = torch.log_softmax(logits, dim=-1)

    token_log_probs = log_probs.gather(
        dim=-1,
        index=labels.unsqueeze(-1),
    ).squeeze(-1)

    answer_start = len(prompt_ids) - 1
    answer_end = answer_start + len(answer_ids)

    answer_log_probs = token_log_probs[
        0,
        answer_start:answer_end,
    ]

    return answer_log_probs.mean().item()

In [41]:
from tqdm.auto import tqdm
import numpy as np

correct = 0
margins = []

for row in tqdm(public_test_quality):
    chosen_score = mean_answer_logprob(
        simpo_model,
        row["prompt"],
        row["chosen"],
    )

    rejected_score = mean_answer_logprob(
        simpo_model,
        row["prompt"],
        row["rejected"],
    )

    if chosen_score > rejected_score:
        correct += 1

    margins.append(chosen_score - rejected_score)

simpo_accuracy = correct / len(public_test_quality)
simpo_mean_margin = float(np.mean(margins))

print("SimPO accuracy:", simpo_accuracy)
print("SimPO mean margin:", simpo_mean_margin)

  0%|          | 0/50 [00:00<?, ?it/s]

SimPO accuracy: 1.0
SimPO mean margin: 1.0226528537273407


In [42]:
SIMPO_PATH = "/kaggle/working/simpo_adapter"

simpo_model.save_pretrained(SIMPO_PATH)
tokenizer.save_pretrained(SIMPO_PATH)

('/kaggle/working/simpo_adapter/tokenizer_config.json',
 '/kaggle/working/simpo_adapter/special_tokens_map.json',
 '/kaggle/working/simpo_adapter/chat_template.jinja',
 '/kaggle/working/simpo_adapter/vocab.json',
 '/kaggle/working/simpo_adapter/merges.txt',
 '/kaggle/working/simpo_adapter/added_tokens.json',
 '/kaggle/working/simpo_adapter/tokenizer.json')